#### ***Loop Engineering***
- ***Designing how an AI agent repeatedly thinks, acts, checks results, learns from errors, and stops only when the task is complete.***

In [1]:
## Non Loop Agent
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    streaming=True,
    temperature=0.3
)

def call_llm(prompt: str):
    response = llm.invoke([
        SystemMessage(content="You are a helpful assistant."),
        HumanMessage(content=prompt)
    ])
    return response.content

In [2]:
task = """
        Write a customer-support reply for the following situation:

        A customer received a damaged laptop.
        Order ID: ORD-2026-101

        Requirements:
        - Apologize to the customer
        - Mention the order ID
        - Offer a replacement
        - Keep the response under 60 words
"""

In [3]:
answer = call_llm(task)
print("Final answer:")
print(answer)

Final answer:
Sorry for the damaged laptop (ORD-2026-101). We'll replace it promptly.


In [5]:
import os
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

MODEL_NAME = "llama-3.3-70b-versatile"
MAX_ITERATIONS = 3
ORDER_ID = "ORD-2026-101"


api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError(
        "Please set the GROQ_API_KEY environment variable."
    )

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.2,
    groq_api_key=api_key,
)

In [6]:
def call_llm(prompt: str) -> str:
    """Generate a customer-support response."""

    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a helpful customer-support agent. "
                    "Follow every requirement carefully."
                )
            ),
            HumanMessage(content=prompt),
        ]
    )
    return response.content.strip()

# Evaluation Harness
def evaluate_answer(answer: str) -> list[str]:
    """
    Check whether the generated answer satisfies all requirements.
    """

    problems = []
    lower_answer = answer.lower()

    # 1. Apology
    apology_words = ["sorry", "apologize", "apologies"]

    if not any(word in lower_answer for word in apology_words):
        problems.append("Add an apology to the customer.")

    # 2. Order ID
    if ORDER_ID not in answer:
        problems.append(f"Mention the order ID {ORDER_ID}.")

    # 3. Replacement
    if "replacement" not in lower_answer:
        problems.append("Clearly offer a replacement.")

    # 4. Word Count
    word_count = len(answer.split())

    if word_count > 60:
        problems.append(
            f"Reduce the response to under 60 words. "
            f"Current length: {word_count} words."
        )
    return problems

In [7]:
task = f"""
        Write a customer-support reply for the following situation.

        Customer received a damaged laptop.

        Order ID: {ORDER_ID}

        Requirements:

        - Apologize to the customer
        - Mention the order ID
        - Offer a replacement
        - Keep the response under 60 words
    """

feedback = "No previous feedback."
previous_answer = ""

In [ ]:
# Loop Engineering
for iteration in range(1, MAX_ITERATIONS + 1):
    print("=" * 60)
    print(f"Iteration {iteration}")
    print("=" * 60)
    prompt = f"""
        Original Task:
        {task}

        Previous Answer:
        {previous_answer or "No previous answer."}

        Feedback:
        {feedback}
        Write an improved customer-support reply.
    """

    # Agent Action
    answer = call_llm(prompt)

    print("\nGenerated Answer:\n")
    print(answer)

    # Harness Evaluation
    problems = evaluate_answer(answer)
    if not problems:
        print("\nEvaluation: PASSED ✅")
        print("\nFinal Answer:\n")
        print(answer)
        break

    print("\nEvaluation: FAILED ❌")
    for problem in problems:
        print("-", problem)

    feedback = "\n".join(f"- {problem}" for problem in problems)
    previous_answer = answer

else:

    print("\nMaximum iterations reached.")

    print("\nBest Available Answer:\n")

    print(previous_answer)

Iteration 1

Generated Answer:

Sorry for the damaged laptop (ORD-2026-101). We'll replace it promptly.

Evaluation: FAILED ❌
- Clearly offer a replacement.
Iteration 2

Generated Answer:

Sorry for the damaged laptop (ORD-2026-101). We'll send a replacement laptop to you.

Evaluation: PASSED ✅

Final Answer:

Sorry for the damaged laptop (ORD-2026-101). We'll send a replacement laptop to you.
